In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    RandomForestClassifier,
)
from sklearn.linear_model import (
    LinearRegression,
    LogisticRegressionCV,
)
from sklearn.metrics import (
    classification_report,
    f1_score,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 2026

np.random.seed(SEED)
rng = np.random.default_rng(SEED)

In [2]:
df = pd.read_csv("hris_performance_train.csv")
# df.head()

In [3]:
# df.describe()

In [4]:
# Train Data

y_class = df["performance_rating"]
X_class = df.drop(columns="performance_rating")
scaler = StandardScaler()
X_class_scaled = scaler.fit_transform(X_class)

y_continuous = df["job_satisfaction"]
X_continuous = df.drop(columns="job_satisfaction")

In [5]:
# Test Data

test_df = pd.read_csv("hris_performance_hidden_test.csv")

y_class_test = test_df["performance_rating"]
X_class_test = test_df.drop(columns="performance_rating")

y_cont_test = test_df["job_satisfaction"]
X_cont_test = test_df.drop(columns="job_satisfaction")

## Regression Task

In [ ]:
# Train

lm = LinearRegression(fit_intercept=True, n_jobs=-1)
lm.fit(X_continuous, y_continuous)
y_hat = lm.predict(X_continuous)
mse = mean_squared_error(y_continuous, y_hat)
r2 = r2_score(y_continuous, y_hat)
print(f"MSE = {mse:.3f} and r^2 = {r2:.3f}")

In [ ]:
# Test

y_hat = lm.predict(X_cont_test)
mse = mean_squared_error(y_cont_test, y_hat)
r2 = r2_score(y_cont_test, y_hat)
print(f"MSE = {mse:.3f} and r^2 = {r2:.3f}")

## Classification Task

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

### Logistic Regression

In [ ]:
# LR Train
# Use LogisticRegressionCV instead of cross_val_score
lr = LogisticRegressionCV(
    penalty="l2",
    Cs=10,
    fit_intercept=True,
    random_state=SEED,
    max_iter=5_000,
    solver="lbfgs",
    n_jobs=-1,
    class_weight="balanced",
    cv=cv,
    scoring="f1_weighted",
)

lr.fit(X_class, y_class)

y_hat = lr.predict(X_class)
# report = classification_report(y_class, y_hat, zero_division=np.nan)
# print(report)
print(f"Weighted f1 score: {f1_score(y_class, y_hat, average='weighted'):.3f}")

In [ ]:
lr.C_

In [ ]:
# LR Test

y_hat = lr.predict(X_class_test)
report = classification_report(y_class_test, y_hat, zero_division=np.nan)
print(report)

### KNN

In [ ]:
# Train
knn = KNeighborsClassifier(n_jobs=-1)
knn.fit(X_class_scaled, y_class)

y_hat = knn.predict(X_class_scaled)
report = classification_report(y_class, y_hat, zero_division=np.nan)
print(f"Full Training Set Report:\n{report}")

cv_scores = cross_val_score(knn, X_class_scaled, y_class, cv=cv, scoring="f1_weighted")
print(f"CV f1s:\n{cv_scores}")

In [ ]:
# Test
X_class_test_scaled = scaler.transform(X_class_test)

y_hat = knn.predict(X_class_test_scaled)
report = classification_report(y_class_test, y_hat)
print(report)

### Decision Tree

In [ ]:
# Train

dt = DecisionTreeClassifier(
    random_state=SEED, class_weight="balanced", ccp_alpha=0.000015
)

dt.fit(X_class, y_class)
y_hat = dt.predict(X_class)
print(f"Full training set f1: {f1_score(y_class, y_hat, average='weighted'):.3f}")

cv_scores = cross_val_score(dt, X_class, y_class, cv=cv, scoring="f1_weighted")
print(f"CV f1s:\n{cv_scores}")

In [ ]:
# Test

y_hat = dt.predict(X_class_test)
print(f"Weighted f1 score: {f1_score(y_class_test, y_hat, average='weighted'):.3f}")

### Random Forests

In [ ]:
# Train
# A callable provided for oob scoring.
def f1_scoring(y_true, y_pred):
    return f1_score(y_true, y_pred, average="weighted")


rf = RandomForestClassifier(
    n_estimators=464,
    random_state=SEED,
    class_weight="balanced",
    max_depth=17,
    max_features=0.5,
    bootstrap=True,
    oob_score=f1_scoring,
    n_jobs=-1,
)
rf.fit(X_class, y_class)
y_hat = rf.predict(X_class)
# report = classification_report(y_class, y_hat, zero_division=np.nan)
# print(f"Full Training Set Report:\n{report}")
print(f"OOB weighted f1 score: {rf.oob_score_:.3f}")

Full Training Set Report:
              precision    recall  f1-score   support

           1       0.99      0.93      0.96      6095
           2       0.94      0.90      0.92     13187
           3       0.95      0.95      0.95     55370
           4       0.87      0.92      0.89     16722
           5       0.91      0.94      0.92      6126

    accuracy                           0.94     97500
   macro avg       0.93      0.93      0.93     97500
weighted avg       0.94      0.94      0.94     97500

OOB weighted f1 score: 0.729


In [ ]:
# Test

y_hat = rf.predict(X_class_test)
# report = classification_report(y_class_test, y_hat, zero_division=np.nan)
# print(report)
print(f"Weighted f1 score: {f1_score(y_class_test, y_hat, average='weighted'):.3f}")

              precision    recall  f1-score   support

           1       0.62      0.28      0.39      2032
           2       0.65      0.55      0.60      4396
           3       0.82      0.88      0.85     18457
           4       0.62      0.72      0.67      5573
           5       0.54      0.42      0.47      2042

    accuracy                           0.74     32500
   macro avg       0.65      0.57      0.60     32500
weighted avg       0.73      0.74      0.73     32500

Weighted f1 score: 0.732
